# Batch-effect reverse-noise -- figures + metrics table

Visualization and metrics for the reverse-noise benchmark, from the saved results. Same layout
as `bifurcation_figs`, pointed at this benchmark's tree: seeds 10-19, ground truth rebuilt by
`RS.make_reverse_noise` with the per-time strength schedule stored in the shared data npz.
`ROOT` below picks where the results are read from (default: the shipped tree).

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
REPO = P.REPO
from uotreg import trajectory_metrics as TM
from uotreg.metrics import w2
import _rev_common as C          # paths + data loader for the REVERSE benchmark
from uotreg import reverse_sim as RS    # the reverse-noise generator

## Parameters
`COLUMNS` is the panel order AND the table order. Each entry says where its trajectory comes from:
`("ours", key_in_the_npz)` or `("baseline", method_folder_name)`.

In [ ]:
DIM      = globals().get("DIM", 10)
# 10-19, deliberately distinct from the bifurcation benchmark's 0-9 so the two never share a seed
SEEDS    = list(C.SEEDS)                      # 10..19; narrow it to speed a re-run up
VIZ_SEED = globals().get("VIZ_SEED", str(C.SEEDS[0]))   # the seed both figures draw
TRAJ_T   = C.TRAJ_T                         # snapshot indices 0..8
# the shipped paper results; switch to P.results(C.RESULTS_DIRNAME) to prefer your own
# `new_results/` re-run of the estimate -> trajectories pipeline
ROOT     = P.shipped(C.RESULTS_DIRNAME)

# text size: printed pt = fontsize x (LaTeX width / figsize width). Same choices as
# `bifurcation_figs` (1x2 at 0.75\linewidth, 1x6 at \linewidth).
FS_DIST = 1.45
FS_TRAJ = 1.85
DIST_FIGSIZE = (7.8, 4.0)
TRAJ_FIGSIZE = (15.0, 4.0)


def _text(fs):
    """Apply one multiplier to every font size at once. Called per figure, since the two canvases
    have different widths and therefore different rescale factors."""
    plt.rcParams.update({"font.size": 11 * fs, "axes.titlesize": 11 * fs, "axes.labelsize": 11 * fs,
                         "figure.titlesize": 13 * fs, "xtick.labelsize": 10 * fs,
                         "ytick.labelsize": 10 * fs, "legend.fontsize": 10 * fs})


# ---- colours (the project's settled palette) ----------------------------------------------------
OBS_C, EST_C = "tab:orange", "tab:purple"    # observed (batch-perturbed) / our estimate
TRUE_C = "0.72"                              # ground truth underneath
PATH_C = "tab:purple"                        # fitted per-cell paths
START_C, END_C = "tab:green", "tab:red"
CURVE_KW = dict(color="tab:blue", ls="--", lw=1.0, alpha=0.65)   # the true branch curves
PENDING_NOTE = "run pending"

N_SHOW_DIST = 300   # cells drawn per time in fig_distributions
MAX_CELLS   = 50  # paths drawn per method in fig_traj_panel
# Axis-box tightness for the 1x6, centred on the true branch curves. <1 crops: at 0.85 MioFlow's
# widest cells fall outside the frame on purpose, so the other five panels are not zoomed out to
# accommodate one method's spray. 1.0 = everything drawn fits.
ZOOM = 0.85

# panel/table order. The `full_` prefix picks the ambient-dim trajectory out of ours' npz;
# every column arrives as (T, N, dim) and is projected by the same call.
MIOFLOW_KEY = "MioFlow"
COLUMNS = [("ours: UOT",   "ours",     "full_ours_UOT_maps"),
           ("ours: OTCFM", "ours",     "full_ours_flow_OT-CFM_unshared"),
           ("WOT",         "baseline", "WOT"),
           ("MMFM",        "baseline", "MMFM"),
           ("MioFlow",     "baseline", MIOFLOW_KEY),
           ("TIGON",       "baseline", "TIGON")]

# The four metrics the paper reports. `w2_10d` is the dict key `w2_path_full`, and `TM.report` only
# fills it when it is handed `trajs_full=` -- otherwise it is silently nan. That is why `metrics()`
# below keeps the full-dim trajectory around instead of projecting once and discarding it.
METRICS = [("adher_indiv", "m.s.e (indiv)"), ("w2_path", "w2_path"),
           ("w2_path_full", f"w2_{DIM}d"), ("roughness", "roughness")]

print(f"batcheffect REVERSE paper figs | d={DIM} seeds={SEEDS} viz_seed={VIZ_SEED} | root={os.path.relpath(ROOT, P.REPO)}")

## Loaders
`ours` trajectories live inside one npz per seed; baselines are one `.npy` each. Everything is
returned full-dim `(T, N, dim)` and projected to the 2-D signal plane only where it is needed.

In [ ]:
_ours_traj_cache, _ours_est_cache = {}, {}


def _ours_npz(seed, what):
    """`what` = "trajs" | "est". None when that file does not exist."""
    cache = _ours_traj_cache if what == "trajs" else _ours_est_cache
    if seed not in cache:
        # the reverse run writes ours' npz FLAT under `results/batcheffectreverse/`, not in an
        # `ours/` subfolder as the bifurcation benchmark does
        p = os.path.join(ROOT, f"reverse_d{DIM}_new_seed{seed}_{what}.npz")
        cache[seed] = np.load(p) if os.path.exists(p) else None
        if cache[seed] is None:
            print(f"   ! missing {os.path.basename(p)}")
    return cache[seed]


def load_traj(seed, kind, key):
    """(T, N, dim) for one (seed, column), or None when that method has nothing saved."""
    if kind == "ours":
        z = _ours_npz(seed, "trajs")
        if z is None:
            return None
        k = key if key in z.files else key.replace("full_", "", 1)   # older npz: projected only
        return np.asarray(z[k], np.float32) if k in z.files else None
    p = C.traj_path(DIM, seed, key, root=ROOT)
    return np.asarray(np.load(p), np.float32) if os.path.exists(p) else None


def est_times(seed):
    """The times `est_series` covers. The reverse export records them (`est_t`); the trajectory grid
    additionally carries the OBSERVED t=0, which has no estimate."""
    z = _ours_npz(seed, "est")
    if z is not None and "est_t" in z.files:
        return [int(t) for t in np.asarray(z["est_t"]).ravel()]
    return [t for t in TRAJ_T][1:]


def load_est(seed):
    """Our estimated distributions `(T, n_gen, dim)` at times `TRAJ_T`, or None."""
    z = _ours_npz(seed, "est")
    return None if z is None else np.asarray(z["est_series"], np.float32)


def rebuild_G(seed, n_per, d=None):
    """The ground-truth generator, rebuilt from the seed -- identical to what every method fitted.

    The reverse-noise simulation is parameterised by `delta_pre` and a per-time `STRENGTH_SCHEDULE`
    (the bifurcation one has a single flat `strength`). Both are echoed into every saved npz, so a
    run made before the knobs were retuned still rebuilds its OWN data -- take them from the npz
    when it has them and fall back to the current defaults only when it does not."""
    d = d or {}
    return RS.make_reverse_noise(seed=int(seed), n_per_time=n_per, dim=DIM,   # ids are strings
                                 delta_pre=d.get("delta_pre", C.DELTA_PRE),
                                 schedule=d.get("schedule", C.STRENGTH_SCHEDULE))


def coverage():
    """Which (seed, method) cells exist. Run this first."""
    print(f"  {'seed':>5}  " + "".join(f"{lab:>14}" for lab, _, _ in COLUMNS))
    have = {}
    for s in SEEDS:
        row = []
        for lab, kind, key in COLUMNS:
            ok = load_traj(s, kind, key) is not None
            have[(s, lab)] = ok
            row.append(f"{'yes':>14}" if ok else f"{'--':>14}")
        print(f"  {s:>5}  " + "".join(row))
    missing = sorted({lab for (s, lab), ok in have.items() if not ok})
    if missing:
        print(f"\n  no files for: {missing}  -> blank column(s) in the panel, '{PENDING_NOTE}' in the table.")
    return have


COV = coverage()

## Section 1: metrics
Every method scored by the same `TM.report`, so the rows are comparable by construction.
`m.s.e (indiv)` = per-cell MSE against the cell's own ground-truth path (requires the saved
trajectory to keep the X0 row order); `w2_path` = W2 predicted-vs-true clouds on the 2-D signal
plane, averaged over times; `w2_10d` = the same in all ambient dimensions; `roughness` = path
wiggle vs net displacement (well below the truth's own roughness = over-smoothed, not better).
Runs before the figures so the panels annotate themselves from the same numbers the table prints.

In [ ]:
def metrics(seeds=None, verbose=True):
    """(rows, agg): per-seed metrics per column, and mean/std over the seeds that have data."""
    seeds = list(seeds or SEEDS)
    rows = {}
    for s in seeds:
        full = {}
        for lab, kind, key in COLUMNS:
            t = load_traj(s, kind, key)
            if t is None:
                continue
            if not np.isfinite(t).all():
                print(f"    ! {lab} seed {s}: non-finite values -- skipped"); continue
            full[lab] = t
        if not full:
            continue
        d = C.load_data(DIM, s, root=ROOT)
        G = rebuild_G(s, len(d["raw_series"][0]), d)
        # 2-D for the projected metrics, ambient-dim for w2_10d -- both from the SAME array.
        proj = {lab: (t if t.shape[-1] == 2 else G.project_traj(t)) for lab, t in full.items()}
        fulld = {lab: t for lab, t in full.items() if t.shape[-1] != 2}
        rows[s] = TM.report(G, proj, TRAJ_T, d["labels0"], verbose=False, trajs_full=fulld)
        if verbose:
            nof = [lab for lab in proj if lab not in fulld]
            print(f"  [seed {s}] scored {len(proj)}"
                  + (f"   ! 2-D only, no w2_{DIM}d: {nof}" if nof else ""))
    agg = {}
    for lab, _, _ in COLUMNS:
        agg[lab] = {}
        for key, _ in METRICS:
            v = [rows[s][lab][key] for s in rows
                 if lab in rows[s] and not (isinstance(rows[s][lab][key], float)
                                            and np.isnan(rows[s][lab][key]))]
            agg[lab][key] = (float(np.mean(v)), float(np.std(v)), len(v)) if v else (None, None, 0)
    return rows, agg


ROWS, AGG = metrics()

## The table

In [ ]:
def table(agg=None, rows=None):
    agg = agg if agg is not None else AGG
    print(f"\n[batch-effect REVERSE-NOISE d={DIM}] mean +/- std over seeds  (lower = better on all four)")
    print(f"  {'method':14s}" + "".join(f"{h:>21}" for _, h in METRICS) + f"{'seeds':>7}")
    best = {}
    for key, _ in METRICS:
        vals = [agg[lab][key][0] for lab, _, _ in COLUMNS if agg[lab][key][0] is not None]
        best[key] = min(vals) if vals else None
    for lab, _, _ in COLUMNS:
        cells = ""
        for key, _ in METRICS:
            m, sd, n = agg[lab][key]
            if m is None:
                cells += f"{'--':>21}"
            else:
                star = "*" if best[key] is not None and m == best[key] else " "
                cells += f"{m:>12.3f}+-{sd:<6.3f}{star}"
        n = max(agg[lab][k][2] for k, _ in METRICS)
        print(f"  {lab:14s}{cells}" + (f"{n:>7}" if n else f"   {PENDING_NOTE}"))
    # the truth's own roughness -- the reference a roughness number is read against
    rows = rows if rows is not None else ROWS
    if rows:
        s0 = sorted(rows)[0]
        d = C.load_data(DIM, s0, root=ROOT)
        G = rebuild_G(s0, len(d["raw_series"][0]), d)
        print(f"\n  * = best in column (among methods present).  truth roughness (seed {s0}) = "
              f"{G.truth_roughness(TRAJ_T):.3f} -- a method far BELOW it is over-smoothed.")
    print(f"  m.s.e (indiv) needs the saved trajectory to keep the X0 row order.")


table()

## Section 2: the distribution figure (1x2)
Left: the batch-perturbed clouds the estimator is given; right: our denoised estimate. Colour =
time (light -> dark), dashed blue = the true branch means; the metric box carries the mean W2 to
the truth.

In [ ]:
def fig_distributions(seed=None, n_show=N_SHOW_DIST, figsize=None,
                      fs=None, w2_box=False):
    """1x2: observed (all times) | our estimate (all times). Returns the figure."""
    seed = VIZ_SEED if seed is None else seed
    fs = FS_DIST if fs is None else fs
    _text(fs)
    d = C.load_data(DIM, seed, root=ROOT)
    G = rebuild_G(seed, len(d['raw_series'][0]), d)
    est = load_est(seed)
    EST_T = est_times(seed)
    if est is None:
        print(f"  ! no estimate saved for seed {seed} -- run Section 1 of reverse_estimate")
        return None

    rng = np.random.default_rng(0)
    sub = lambda X: (X if len(X) <= n_show else X[rng.choice(len(X), n_show, replace=False)])

    fig, axes = plt.subplots(1, 2, figsize=figsize or DIST_FIGSIZE, dpi=140, sharex=True, sharey=True)
    # Same colour scheme as `plotting.show_noisy_data` / `show_estimates`, just with every time in
    # ONE pair of axes: clean truth grey underneath, observed orange, estimate purple. Time is read
    # off the geometry (left = early, right = late), so it needs no colour of its own.
    # `est_series` holds the ESTIMATED clouds only. On this benchmark `TRAJ_T` is 0..8 and t=0 is the
    # OBSERVED starting cloud, so there are 9 trajectory times but only 8 estimates -- indexing the
    # estimates by `len(TRAJ_T)` walks off the end (it does not on the bifurcation benchmark, whose
    # TRAJ_T is 1..8). Iterate the estimates themselves.
    panels = [("observed", [np.asarray(r) for r in d["raw_series"]], OBS_C),
              ("UOTReg estimate", [np.asarray(e) for e in est], EST_C)]
    for ax, (ttl, series, col) in zip(axes, panels):
        for t in TRAJ_T:                                          # clean truth underneath, all times
            q = C.project2d(sub(np.asarray(G.truth(t))), DIM)
            ax.scatter(q[:, 0], q[:, 1], s=4, c=TRUE_C, alpha=0.35, linewidths=0)
        for cloud in series:
            p = C.project2d(sub(cloud), DIM)
            ax.scatter(p[:, 0], p[:, 1], s=4, color=col, alpha=0.5, linewidths=0)
        ax.plot(G.curves[:, 0, 0], G.curves[:, 0, 1], **CURVE_KW)
        ax.plot(G.curves[:, 1, 0], G.curves[:, 1, 1], **CURVE_KW)
        ax.set_title(ttl, fontsize=10 * fs)      # a touch under the body size: these two labels
    #                                              say what the panel is, they are not headings
        ax.set_xlabel("signal 1", fontsize=11 * fs)
    axes[0].set_ylabel("signal 2", fontsize=11 * fs)

    # mean over times of W2(cloud, truth) -- one number per panel, same scale for both. ALWAYS
    # computed and printed: as of 2026-08-20 it belongs in the CAPTION, not in the panels, matching
    # `metrics_and_viz/bifurcation_figs.py` so the two supplement figures stay a matched pair.
    if True:
        w_obs = float(np.mean([w2(np.asarray(d["raw_series"][i], np.float32),
                                  np.asarray(G.truth(t), np.float32)) for i, t in enumerate(TRAJ_T)]))
        # the estimates cover `EST_T` (1..8), not `TRAJ_T` (0..8) -- pair each estimate with the
        # truth AT ITS OWN TIME, so the two panels' numbers stay comparable and nothing runs off the
        # end of the array
        w_est = float(np.mean([w2(np.asarray(e, np.float32),
                                  np.asarray(G.truth(t), np.float32))
                               for e, t in zip(est, EST_T)]))
        for ax, val in zip(axes, (w_obs, w_est)):
            if not w2_box:
                continue
            ax.text(0.03, 0.06, f"mean W2 to truth = {val:.2f}", transform=ax.transAxes,
                    fontsize=10 * fs, family="monospace",
                    bbox=dict(boxstyle="round", fc="white", ec="0.6", alpha=0.85))
        print(f"  [seed {seed}] mean W2 to truth -> FOR THE CAPTION: "
              f"observed {w_obs:.2f}, estimate {w_est:.2f}")

    # header stack: suptitle -> colour key -> panel titles -> panels
    fig.tight_layout(rect=[0, 0, 1, 0.86])
    fig.suptitle(f"batch-effect simulation, reverse noise (seed {seed})",
                 fontsize=12 * fs, y=0.99)
    fig.legend(handles=[
        Line2D([0], [0], marker="o", ls="", mfc=OBS_C, mec="none", label="observed"),
        Line2D([0], [0], marker="o", ls="", mfc=EST_C, mec="none", label="estimate"),
        Line2D([0], [0], marker="o", ls="", mfc=TRUE_C, mec="none", label="clean truth"),
        Line2D([0], [0], **{**CURVE_KW, "lw": 1.4}, label="true branch means")],
        loc="upper center", ncol=4, frameon=False, fontsize=10 * fs,
        handletextpad=0.3, columnspacing=1.1, bbox_to_anchor=(0.5, 0.93))
    plt.show()
    return fig


fig_distributions()

## Section 3: the trajectory panel (1x6)
One method per box, in `COLUMNS` order; each x-label carries that method's aggregate metric. A
method with nothing saved keeps its box, labelled `run pending`.

In [ ]:
def _square_lims(G, trajs, pad=0.08, zoom=None):
    """Axis box square in data units, centred on the true branch curves (so one spraying method
    cannot drag the frame), then scaled by `zoom` (<1 crops the widest method on purpose)."""
    zoom = ZOOM if zoom is None else zoom
    cx = float(np.mean([G.curves[:, :, 0].min(), G.curves[:, :, 0].max()]))
    cy = float(np.mean([G.curves[:, :, 1].min(), G.curves[:, :, 1].max()]))
    reach = [float(np.abs(G.curves[:, :, 0] - cx).max()), float(np.abs(G.curves[:, :, 1] - cy).max())]
    for t in trajs:
        reach += [float(np.abs(t[..., 0] - cx).max()), float(np.abs(t[..., 1] - cy).max())]
    half = max(reach) * (1 + pad) * zoom
    return (cx - half, cx + half), (cy - half, cy + half)


def fig_traj_panel(seed=None, max_cells=MAX_CELLS, figsize=None, fs=None,
                   columns=None, annotate=None, lim=None, zoom=None):
    """1x6 (one box per method). The figure carries no numbers — the table does.

    No observed-cell background: with six small boxes the grey haze only competed with the paths.
    The dashed true branch means stay, and they are the reference the paths are read against.

    `annotate="w2_path"` (or any METRICS key) puts that metric back under each panel.
    `zoom` (default `ZOOM`) tightens the shared axis box; `lim=((x0,x1),(y0,y1))` overrides it."""
    seed = VIZ_SEED if seed is None else seed
    fs = FS_TRAJ if fs is None else fs
    cols_ = columns or COLUMNS
    _text(fs)
    d = C.load_data(DIM, seed, root=ROOT)
    G = rebuild_G(seed, len(d['raw_series'][0]), d)
    lab_of = dict((k, h) for k, h in METRICS)

    drawn = []
    for lab, kind, key in cols_:
        t = load_traj(seed, kind, key)
        if t is not None:
            drawn.append(t if t.shape[-1] == 2 else G.project_traj(t))
    box = lim or _square_lims(G, drawn, zoom=zoom)

    fig, axes = plt.subplots(1, len(cols_), figsize=figsize or TRAJ_FIGSIZE, dpi=140,
                             sharex=True, sharey=True)
    for ax, (lab, kind, key) in zip(np.atleast_1d(axes), cols_):
        ax.plot(G.curves[:, 0, 0], G.curves[:, 0, 1], **CURVE_KW)
        ax.plot(G.curves[:, 1, 0], G.curves[:, 1, 1], **CURVE_KW)
        t = load_traj(seed, kind, key)
        if t is None:
            ax.text(0.5, 0.5, PENDING_NOTE, transform=ax.transAxes, ha="center", va="center",
                    fontsize=10 * fs, color="0.35",
                    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.7", alpha=0.9))
        else:
            tr2 = t if t.shape[-1] == 2 else G.project_traj(t)
            for c in range(min(max_cells, tr2.shape[1])):
                ax.plot(tr2[:, c, 0], tr2[:, c, 1], "-", color=PATH_C, lw=0.6, alpha=0.55)
            ax.scatter(tr2[0, :, 0], tr2[0, :, 1], s=8, c=START_C, zorder=3, linewidths=0)
            ax.scatter(tr2[-1, :, 0], tr2[-1, :, 1], s=8, c=END_C, zorder=3, linewidths=0)
            m = AGG.get(lab, {}).get(annotate, (None,))[0] if annotate else None
            if m is not None:
                ax.set_xlabel(f"{lab_of.get(annotate, annotate)} {m:.2f}", fontsize=10 * fs)
        ax.set_title(lab, fontsize=11 * fs)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlim(*box[0]); ax.set_ylim(*box[1])
        ax.set_aspect("equal", adjustable="box")     # square data units -> square boxes

    # header stack, top to bottom: suptitle -> legend row -> panel titles -> panels. `rect` reserves
    # the band; the legend sits inside it. Keep the three y-values in step if you change the height:
    # `rect`'s top is a FRACTION, so shortening the figure (as removing the metric x-label did) moves
    # the band's inches down and the titles ride up into the legend.
    fig.tight_layout(rect=[0, 0, 1, 0.650])
    # Same convention as `bifurcation_figs.py` and the real-data figures: "inferred trajectories in
    # <what> (d = N)", no seed, no colon. "reversed noise schedule" rather than "reverse noise",
    # because the GEOMETRY is the unchanged bifurcation -- only the noise schedule runs backwards.
    fig.suptitle(f"inferred trajectories in the batch-effect simulation, reversed noise schedule "
                 f"(d = {DIM})", fontsize=12 * fs, y=0.935)
    fig.legend(handles=[
        Line2D([0], [0], marker="o", ls="", mfc=START_C, mec="none", label=f"start (t={TRAJ_T[0]})"),   # 0 here, 1 on the bifurcation grid
        Line2D([0], [0], marker="o", ls="", mfc=END_C, mec="none", label=f"end (t={TRAJ_T[-1]})"),
        Line2D([0], [0], color=PATH_C, lw=1.4, label="fitted path"),
        Line2D([0], [0], **{**CURVE_KW, "lw": 1.4}, label="true branch means")],
        loc="upper center", ncol=4, frameon=False, fontsize=10 * fs,
        handletextpad=0.3, columnspacing=1.2, bbox_to_anchor=(0.5, 0.845))
    plt.show()
    return fig


fig_traj_panel()